## Step up cryptographic environment

In [1]:
%pip install cryptography

In [2]:
import os
import json
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidTag


In [3]:
import cryptography

print("Cryptography version:", cryptography.__version__)
print("AES-GCM ready")

Cryptography version: 50.0.2
AES-GCM ready


## Defining the encryption protocol

In [4]:
class HealthcareProtocol:

    def __init__(self, flawed=False):

        self.key = AESGCM.generate_key(bit_length=256)
        self.aesgcm = AESGCM(self.key)

        self.counter = 0
        self.flawed = flawed

    def encrypt(self, plaintext, metadata):

        if self.counter >= 2**96:
            raise ValueError("Nonce space exhausted")

        # Reusing nonce in the flawed implementation
        nonce_number = 0 if self.flawed else self.counter

        nonce = nonce_number.to_bytes(12, "big")

        self.counter += 1

        aad = json.dumps(
            metadata, sort_keys=True
        ).encode("utf-8")

        encrypted = self.aesgcm.encrypt(
            nonce,
            plaintext.encode("utf-8"),
            aad
        )

        return {
            "nonce": nonce,
            "ciphertext": encrypted[:-16],
            "tag": encrypted[-16:],
            "aad": aad
        }

    def decrypt(self, packet):

        plaintext = self.aesgcm.decrypt(
            packet["nonce"],
            packet["ciphertext"] + packet["tag"],
            packet["aad"]
        )

        return plaintext.decode("utf-8")

### Patient metadata

In [5]:
metadata = {
    "patient_id": "P1001",
    "sender": "Hospital_A",
    "recipient": "GP_B",
    "key_version": 1
}

### Testing secure encryption

In [6]:
secure = HealthcareProtocol(flawed=False)

patient_record = "Patient requires follow-up appointment"

packet = secure.encrypt(patient_record, metadata)

print("Nonce:", packet["nonce"].hex())
print("Ciphertext:", packet["ciphertext"].hex())
print("Authentication Tag:", packet["tag"].hex())

decrypted = secure.decrypt(packet)

print("Decrypted Record:", decrypted)
print("Successful:", decrypted == patient_record)

Nonce: 000000000000000000000000
Ciphertext: e2905f799591deeae48923f2123a977495f5c0fdd11bd889f7844f7b0a503fcc3b3ec94bd740
Authentication Tag: 1aade561147cae92b868e07970ebda8e
Decrypted Record: Patient requires follow-up appointment
Successful: True


## Security Testing
### Creating test data

In [7]:
test_record = "Patient requires urgent medical review"

test_packet = secure.encrypt(test_record, metadata)

print("Test packet generated")

Test packet generated


### Testing successful decryption

In [8]:
result = secure.decrypt(test_packet)

print("Original:", test_record)
print("Decrypted:", result)
print("Test passed:", result == test_record)

Original: Patient requires urgent medical review
Decrypted: Patient requires urgent medical review
Test passed: True


### Testing modified ciphertext

In [9]:
tampered_packet = test_packet.copy()

tampered_ciphertext = bytearray(tampered_packet["ciphertext"])
tampered_ciphertext[0] ^= 1

tampered_packet["ciphertext"] = bytes(tampered_ciphertext)

try:
    secure.decrypt(tampered_packet)
    print("Test failed: Modified ciphertext accepted")

except InvalidTag:
    print("Test passed: Modified ciphertext rejected")

Test passed: Modified ciphertext rejected


### Testing modified metadata

In [10]:
modified_packet = test_packet.copy()

modified_packet["aad"] = json.dumps({
    "patient_id": "P2002",
    "sender": "Hospital_A",
    "recipient": "GP_B",
    "key_version": 1
}, sort_keys=True).encode("utf-8")

try:
    secure.decrypt(modified_packet)
    print("Test failed: Modified metadata accepted")

except InvalidTag:
    print("Test passed: Modified metadata rejected")

Test passed: Modified metadata rejected


### Testing incorrect key

In [11]:
wrong_key = AESGCM.generate_key(bit_length=256)
wrong_aesgcm = AESGCM(wrong_key)

try:
    wrong_aesgcm.decrypt(
        test_packet["nonce"],
        test_packet["ciphertext"] + test_packet["tag"],
        test_packet["aad"]
    )

    print("Test failed: Incorrect key accepted")

except InvalidTag:
    print("Test passed: Incorrect key rejected")

Test passed: Incorrect key rejected


### Testing results

In [12]:
print("SECURITY TEST SUMMARY")
print("---------------------")
print("Successful decryption: Expected to pass")
print("Modified ciphertext: Expected rejection")
print("Modified metadata: Expected rejection")
print("Incorrect key: Expected rejection")

SECURITY TEST SUMMARY
---------------------
Successful decryption: Expected to pass
Modified ciphertext: Expected rejection
Modified metadata: Expected rejection
Incorrect key: Expected rejection


Creating the flawed implementation

In [13]:
flawed = HealthcareProtocol(flawed=True)

print("Flawed implementation created")
print("Nonce reuse enabled:", flawed.flawed)

Flawed implementation created
Nonce reuse enabled: True


Creating two patient record

In [14]:
known_plaintext = "Patient record: NORMAL"
secret_plaintext = "Patient record: SECRET"

print("Known plaintext:", known_plaintext)
print("Secret plaintext: [HIDDEN]")

Known plaintext: Patient record: NORMAL
Secret plaintext: [HIDDEN]


Encrypting both records

In [15]:
packet1 = flawed.encrypt(known_plaintext, metadata)
packet2 = flawed.encrypt(secret_plaintext, metadata)

print("First nonce:", packet1["nonce"].hex())
print("Second nonce:", packet2["nonce"].hex())

print("First ciphertext:", packet1["ciphertext"].hex())
print("Second ciphertext:", packet2["ciphertext"].hex())

First nonce: 000000000000000000000000
Second nonce: 000000000000000000000000
First ciphertext: 81e19bfaced416ea00196e27602e8230205a74fffd49
Second ciphertext: 81e19bfaced416ea00196e27602e82303d5065e0f951


Implementing the XOR attack

In [16]:
def xor_bytes(a, b):
    return bytes(x ^ y for x, y in zip(a, b))


def recover_plaintext(ciphertext1, ciphertext2, known_text):

    known_bytes = known_text.encode("utf-8")

    recovered = xor_bytes(
        xor_bytes(ciphertext1, ciphertext2),
        known_bytes
    )

    return recovered

Executing the attack

In [17]:
recovered_plaintext = recover_plaintext(
    packet1["ciphertext"],
    packet2["ciphertext"],
    known_plaintext
)

print("Known plaintext:", known_plaintext)
print("Recovered plaintext:", recovered_plaintext.decode("utf-8"))
print("Actual secret plaintext:", secret_plaintext)

print(
    "Attack successful:",
    recovered_plaintext == secret_plaintext.encode("utf-8")
)

Known plaintext: Patient record: NORMAL
Recovered plaintext: Patient record: SECRET
Actual secret plaintext: Patient record: SECRET
Attack successful: True


Running the same attack against both

In [18]:
def run_attack_test(protocol, label):

    first = protocol.encrypt(known_plaintext, metadata)
    second = protocol.encrypt(secret_plaintext, metadata)

    recovered = recover_plaintext(
        first["ciphertext"],
        second["ciphertext"],
        known_plaintext
    )

    success = recovered == secret_plaintext.encode("utf-8")

    print("\n", label)
    print("Nonce 1:", first["nonce"].hex())
    print("Nonce 2:", second["nonce"].hex())
    print("Recovered plaintext:", recovered.decode("utf-8", errors="replace"))
    print("Attack successful:", success)

    return success

Comparative attack results

In [19]:
flawed_result = run_attack_test(
    HealthcareProtocol(flawed=True),
    "FLAWED IMPLEMENTATION"
)

secure_result = run_attack_test(
    HealthcareProtocol(flawed=False),
    "SECURE IMPLEMENTATION"
)

print("\nFINAL COMPARISON")
print("----------------")
print("Flawed attack succeeded:", flawed_result)
print("Secure attack succeeded:", secure_result)

assert flawed_result is True
assert secure_result is False

print("\nBoth comparative tests passed")


 FLAWED IMPLEMENTATION
Nonce 1: 000000000000000000000000
Nonce 2: 000000000000000000000000
Recovered plaintext: Patient record: SECRET
Attack successful: True

 SECURE IMPLEMENTATION
Nonce 1: 000000000000000000000000
Nonce 2: 000000000000000000000001
Recovered plaintext: �� gC�m�	9����}q��!#
Attack successful: False

FINAL COMPARISON
----------------
Flawed attack succeeded: True
Secure attack succeeded: False

Both comparative tests passed


## Experimental results

In [20]:
print("CRYPTOGRAPHIC ATTACK SUMMARY")
print("=" * 40)

print("Protocol: AES-256-GCM")
print("Vulnerability: Nonce reuse")
print("Attack: Known-plaintext XOR attack")

print("\nFlawed implementation:")
print("Nonce reused: Yes")
print("Plaintext recovered: Yes")

print("\nSecure implementation:")
print("Nonce reused: No")
print("Plaintext recovered: No")

print("\nConclusion:")
print("Nonce reuse compromises confidentiality.")

CRYPTOGRAPHIC ATTACK SUMMARY
Protocol: AES-256-GCM
Vulnerability: Nonce reuse
Attack: Known-plaintext XOR attack

Flawed implementation:
Nonce reused: Yes
Plaintext recovered: Yes

Secure implementation:
Nonce reused: No
Plaintext recovered: No

Conclusion:
Nonce reuse compromises confidentiality.


In [21]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Creating persistent storage


In [22]:
import sqlite3
import os

DB_PATH = "/content/drive/MyDrive/CA2_nonce_state.db"

connection = sqlite3.connect(DB_PATH)

connection.execute("""
CREATE TABLE IF NOT EXISTS nonce_state (
    key_id TEXT PRIMARY KEY,
    next_counter INTEGER NOT NULL
)
""")

connection.commit()
connection.close()

print("Persistent nonce database ready")

Persistent nonce database ready


In [23]:
KEY_PATH = "/content/drive/MyDrive/CA2_demo_key.bin"

if os.path.exists(KEY_PATH):
    with open(KEY_PATH, "rb") as f:
        persistent_key = f.read()
else:
    persistent_key = AESGCM.generate_key(bit_length=256)

    with open(KEY_PATH, "xb") as f:
        f.write(persistent_key)

print("Persistent demonstration key loaded")

Persistent demonstration key loaded


Defining the persistent nonce manager

In [24]:
import hashlib

class PersistentNonceManager:

    def __init__(self, key, database_path):
        self.database_path = database_path
        self.key_id = hashlib.sha256(key).hexdigest()

    def next_nonce(self):

        with sqlite3.connect(self.database_path, timeout=30) as db:

            db.execute("BEGIN IMMEDIATE")

            row = db.execute(
                "SELECT next_counter FROM nonce_state WHERE key_id = ?",
                (self.key_id,)
            ).fetchone()

            counter = row[0] if row else 0

            if counter >= 2**96:
                raise ValueError("Nonce space exhausted")

            next_counter = counter + 1

            db.execute("""
                INSERT INTO nonce_state (key_id, next_counter)
                VALUES (?, ?)
                ON CONFLICT(key_id)
                DO UPDATE SET next_counter = excluded.next_counter
            """, (self.key_id, next_counter))

            db.commit()

        return counter.to_bytes(12, "big")

Testing persistent nonce generation

In [25]:
nonce_manager = PersistentNonceManager(
    persistent_key,
    DB_PATH
)

nonce1 = nonce_manager.next_nonce()
nonce2 = nonce_manager.next_nonce()

print("Nonce 1:", nonce1.hex())
print("Nonce 2:", nonce2.hex())

print("Unique nonces:", nonce1 != nonce2)

Nonce 1: 000000000000000000000000
Nonce 2: 000000000000000000000001
Unique nonces: True


## Replay protection

In [26]:
connection = sqlite3.connect(DB_PATH)

connection.execute("""
CREATE TABLE IF NOT EXISTS replay_state (
    sender TEXT PRIMARY KEY,
    last_sequence INTEGER NOT NULL
)
""")

connection.commit()
connection.close()

print("Replay protection database ready")

Replay protection database ready


Defining replay verification

In [27]:
class ReplayProtection:

    def __init__(self, database_path):
        self.database_path = database_path

    def accept(self, sender, sequence):

        with sqlite3.connect(
            self.database_path, timeout=30
        ) as db:

            db.execute("BEGIN IMMEDIATE")

            row = db.execute(
                "SELECT last_sequence FROM replay_state WHERE sender = ?",
                (sender,)
            ).fetchone()

            last = row[0] if row else -1

            if sequence <= last:
                db.rollback()
                return False

            db.execute("""
                INSERT INTO replay_state (sender, last_sequence)
                VALUES (?, ?)
                ON CONFLICT(sender)
                DO UPDATE SET last_sequence = excluded.last_sequence
            """, (sender, sequence))

            db.commit()
            return True

Testing replay protection

In [28]:
replay = ReplayProtection(DB_PATH)

print("First message:", replay.accept("Hospital_A", 1))
print("Second message:", replay.accept("Hospital_A", 2))
print("Replayed message:", replay.accept("Hospital_A", 1))

First message: True
Second message: True
Replayed message: False


### Integrating the secure protocol

In [29]:

class SecureHealthcareExchange:
    def __init__(self, key, database_path):
        self.aesgcm = AESGCM(key)
        self.nonce_manager = PersistentNonceManager(key, database_path)
        self.replay = ReplayProtection(database_path)

    def encrypt(self, plaintext, metadata):
        nonce = self.nonce_manager.next_nonce()

        sequence = int.from_bytes(nonce, "big")
        authenticated_metadata = {
            **metadata,
            "sequence": sequence
        }

        aad = json.dumps(
            authenticated_metadata,
            sort_keys=True
        ).encode("utf-8")

        ciphertext = self.aesgcm.encrypt(
            nonce,
            plaintext.encode("utf-8"),
            aad
        )

        return {
            "nonce": nonce,
            "ciphertext": ciphertext,
            "aad": aad
        }

    def receive(self, packet):
        # Verify the authentication tag before accepting the message
        plaintext = self.aesgcm.decrypt(
            packet["nonce"],
            packet["ciphertext"],
            packet["aad"]
        )

        metadata = json.loads(packet["aad"].decode("utf-8"))

        accepted = self.replay.accept(
            metadata["sender"],
            metadata["sequence"]
        )

        if not accepted:
            raise ValueError("Replay detected")

        return plaintext.decode("utf-8")


Test secure record exchange

In [31]:

key_path = "/content/drive/MyDrive/CA2_demo_key.bin"

with open(key_path, "rb") as key_file:
    demo_key = key_file.read()

print("Demo key loaded:", len(demo_key) == 32)


Demo key loaded: True


est secure record exchange

In [32]:

database_path = "/content/drive/MyDrive/CA2_nonce_state.db"

exchange = SecureHealthcareExchange(
    demo_key,
    database_path
)

metadata = {
    "patient_id": "P3001",
    "sender": "Hospital_C",
    "recipient": "GP_D",
    "key_version": 1
}

packet = exchange.encrypt(
    "Patient requires specialist consultation",
    metadata
)

received_record = exchange.receive(packet)

print("Original record: Patient requires specialist consultation")
print("Received record:", received_record)
print("Successful exchange:", received_record == "Patient requires specialist consultation")


Original record: Patient requires specialist consultation
Received record: Patient requires specialist consultation
Successful exchange: True


In [33]:

try:
    exchange.receive(packet)
except ValueError as error:
    print("Replay test:", error)


Replay test: Replay detected


Testing metadata tampering

In [34]:

tampered_packet = packet.copy()

tampered_metadata = json.loads(
    tampered_packet["aad"].decode("utf-8")
)

tampered_metadata["recipient"] = "Unauthorised_GP"

tampered_packet["aad"] = json.dumps(
    tampered_metadata,
    sort_keys=True
).encode("utf-8")

try:
    exchange.receive(tampered_packet)
except InvalidTag:
    print("Tampering test: Rejected successfully")


Tampering test: Rejected successfully


## Final validation

In [35]:

print("FINAL PROJECT VALIDATION")
print("=" * 35)

assert decrypted == patient_record
print("AES-GCM encryption: PASSED")

assert flawed_result is True
assert secure_result is False
print("Nonce reuse attack comparison: PASSED")

assert nonce1 != nonce2
print("Persistent nonce generation: PASSED")

assert replay.accept("Hospital_Final_Test", 1) is True
assert replay.accept("Hospital_Final_Test", 1) is False
print("Replay protection: PASSED")

assert received_record == "Patient requires specialist consultation"
print("Integrated record exchange: PASSED")

print("\nALL VALIDATION TESTS PASSED")


FINAL PROJECT VALIDATION
AES-GCM encryption: PASSED
Nonce reuse attack comparison: PASSED
Persistent nonce generation: PASSED
Replay protection: PASSED
Integrated record exchange: PASSED

ALL VALIDATION TESTS PASSED
